# Neural Networks and Deep Learning, MDS HSE

## Homework 3. Gen models.

### General Information

### Grading and Penalties

The maximum possible grade for the assignment without bonuses is 10 points. Submitting the work after the hard deadline is not allowed.

Submitting after the soft deadline incurs a penalty of -1 point per day. Twice per semester (two modules), students are allowed to use an extension and submit by the hard deadline without penalty.

The assignment must be completed individually. “Similar” solutions will be considered plagiarism, and all involved students (including those whose work was copied) will receive no more than 0 points for the assignment. If you find a solution (or part of it) to any task from an open source, you must include a link to that source in a separate section at the end of your work (most likely you won’t be the only one who found it, so providing the link helps avoid suspicion of plagiarism).

Inefficient code implementation may negatively affect your grade. The grade may also be reduced for poorly readable code and poorly formatted plots. All answers must be accompanied by either code or comments explaining how they were obtained.

Use of generative models is allowed under the following conditions:
- The amount of code generated by such models does not exceed 30% of the total.
- You specify the model used and the prompt.
- At the end of your work, you include a **reflection on your experience using generative AI for this homework:  
  Describe how often you had to fix the code yourself or ask the model to correct something. Was it faster than writing the code on your own?

If these requirements are not met, the assignment will not be graded, and the maximum possible score is 0 points.

### About the Assignment

## Warning: the homework assignment is somewhat tedious because it requires training four types of models. But at the same time, it is simple, so there should be no problems with writing the code.

# Introduction

## MAGIC – Major Atmospheric Gamma Imaging Cherenkov Telescope

MAGIC (Major Atmospheric Gamma Imaging Cherenkov) is a system consisting of two Cherenkov telescopes with a diameter of 17 m. They are designed to observe gamma rays from galactic and extragalactic sources in the very high energy range (from 30 GeV to 100 TeV). 

The MAGIC telescopes are currently operated by about 165 astrophysicists from 24 organizations and consortia in 12 countries. MAGIC has enabled the discovery and study of new classes of gamma-ray sources, such as pulsars and gamma-ray bursts (GRBs).

<center><img src="img/magic1.jpg" width="1000"></center>

Source: https://magic.mpp.mpg.de/

Youtube video: https://youtu.be/mjcDSR2vSU8

## Particles from space

Cosmic particles, $\gamma$-quanta (photons) and hadrons (protons), interact with the atmosphere and generate showers of secondary particles. Moving at near-light speeds, these particles emit Cherenkov radiation. Telescopes photograph this radiation. The photographs can be used to determine the type of particle from space: photon or proton.

<center><img src="img/shower.jpg" width="500"></center>

## Photos

The purpose of an atmospheric Cherenkov telescope is to obtain an image of a shower by measuring the Cherenkov light from the shower particles. This image is a geometric projection of the shower onto the detector. Image parameters, or so-called Hillas parameters, were introduced to analyze these images. There are two types of image parameters: shape parameters and orientation parameters. (Source: http://ihp-lx.ethz.ch/Stamet/magic/parameters.html)

<center><img src="img/geo.jpg" width="400"></center>

## Photons vs Hadrons

Images for $\gamma$-quanta (photons) and hadrons (protons) differ in the shape of their clusters. Astronomers use machine learning models to classify these images. To train the models, scientists artificially generate such images for each type of particle using complex physical simulators.

<center><img src="img/gamma_p.png" width="600"></center>

## Simulation acceleration

Complex physical simulators require significant computing resources. They simulate the arrival of particles from space, their interaction with the atmosphere, the formation of rain showers, Cherenkov radiation, and the operation of telescopes to obtain images. However, we can utilize generative adversarial networks for rapid simulation.

In [ ]:
!pip install diffusers

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import DDPMScheduler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

# Data

We will use data from the MAGIC telescope from the UCI repository https://archive.ics.uci.edu/ml/datasets/MAGIC+Gamma+Telescope. Each object in the data is the parameters of a single cluster image and the label of that cluster (photon or hadron):


0. Length: major axis of ellipse [mm]
1. Width: minor axis of ellipse [mm]
2. Size: 10-log of sum of content of all pixels [in #phot]
3. Conc: ratio of sum of two highest pixels over fSize [ratio]
4. Conc1: ratio of highest pixel over fSize [ratio]
5. Asym: distance from highest pixel to center, projected onto major axis [mm]
6. M3Long: 3rd root of third moment along major axis [mm]
7. M3Trans: 3rd root of third moment along minor axis [mm]
8. Alpha: angle of major axis with vector to origin [deg]
9. Dist: distance from origin to center of ellipse [mm]
10. class: g,h # gamma (signal), hadron (background)

In [ ]:
# read data
names = np.array(
    [
        "Length",
        "Width",
        "Size",
        "Conc",
        "Conc1",
        "Asym",
        "M3Long",
        "M3Trans",
        "Alpha",
        "Dist",
        "class",
    ]
)
data = pd.read_csv("magic04.data", header=None)
data.columns = names
data.head()

# Problem statement

Your task is to use generative adversarial networks to learn how to generate cluster parameters on telecope images for each type of particle (photon or hadron):

- $X$ - a matrix of real objects that need to be generated;
- $y$ - class labels that will be used as a condition during generation.


In [ ]:
# cluster parameters in images
X = data[names[:-1]].values
X = np.abs(X)

# Class labels
labels = data[names[-1]].values
y = np.ones((len(labels), 1))
y[labels == "h"] = 0

In [ ]:
# Examples
X[:2]

In [ ]:
# Examples
y[:10]

# Data visualization

Each image is described by 10 parameters. Let's construct distributions of values for each parameter for each particle type.

In [ ]:
def plot_hists(X1, X2, names, label1, label2, bins=np.linspace(-3, 3, 61)):
    plt.figure(figsize=(4 * 4, 4 * 2))
    for i in range(X1.shape[1]):
        plt.subplot(3, 4, i + 1)
        plt.hist(X1[:, i], bins=bins, alpha=0.5, label=label1, color="C0")
        plt.hist(X2[:, i], bins=bins, alpha=0.5, label=label2, color="C1")
        plt.xlabel(names[i], size=14)
        plt.legend(loc="best")
    plt.tight_layout()

In [ ]:
plot_hists(
    X[y[:, 0] == 0], X[y[:, 0] == 1], names, label1="Hadrons", label2="Photons", bins=50
)

# Data preprocessing

The graph shows that the distributions for many features have heavy tails. This makes training generative models more difficult. Therefore, we need to transform the data in some way to remove these heavy tails. 

## Task 1 (0.5 points)

Use the `sklearn.preprocessing.QuantileTransformer` function to transform the input data `X`. This transformation ensures that the distribution of each parameter is normal. A description of the function is available at http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html. Use the parameter value `output_distribution=‘normal’`. 

In [ ]:
### YOUR CODE IS HERE ######
X_qt = ...
### THE END OF YOUR CODE ###

In [ ]:
plot_hists(
    X_qt[y[:, 0] == 0],
    X_qt[y[:, 0] == 1],
    names,
    label1="Hadrons",
    label2="Photons",
    bins=50,
)

# Training and test samples

In [ ]:
# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_qt, y, stratify=y, test_size=0.5, shuffle=True, random_state=11
)

In [ ]:
plot_hists(X_train, X_test, names, label1="Train", label2="Test")

# Conditional WGAN

We will use Conditional WGAN, which is shown in the figure. As the condition y, we will use the class label: **0** - hadron, **1** - photon. Thus, we will tell the generator which particle to generate image parameters for.  

<center><img src="img/cgan.png" width="800"></center>

The generator $\hat{x} = G(z, y)$ will take the noise vector $z$ and the condition vector $y$ as input and output the generated (fake) parameter vector $\hat{x}$. 

The discriminator $D(x, y)$ will accept the parameter vector $x$ and the condition vector $y$ as input and return a rational number.

We will train `Conditional WGAN` with the following loss function:

$$L(G, D) = -\frac{1}{n} \sum_{x_i \in X, y_i \in y} D(x_i, y_i) + \frac{1}{n} \sum_{z_i \in Z, y_i \in y} D(G(z_i, y_i), y_i) \to \max_G \min_D$$

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DEVICE

## Task 2 (0.25 points)

Implement a neural network for a generator with the following layers:
- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Output layer.

Hint: use the `nn.Sequential()` function.

In [ ]:
class Generator(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()

        ### YOUR CODE IS HERE ######
        self.net = ...
        ### THE END OF YOUR CODE ###

    def forward(self, z, y):
        zy = torch.cat((z, y), dim=1)
        return self.net(zy)

## Task 3 (0.25 points)

Implement a neural network for a discriminator with the following layers:
- Fully connected layer with 100 neurons;
- ReLU activation function;
- Fully connected layer with 100 neurons;
- ReLU activation function;
- Output layer.

Hint: use the `nn.Sequential()` function.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()

        ### YOUR CODE IS HERE ######
        self.net = ...
        ### THE END OF YOUR CODE ###

    def forward(self, x, y):
        xy = torch.cat((x, y), dim=1)
        return self.net(xy)

## Task 4 (1 point)

Implement a class for training a generative model.

- Hint 1: Don't forget to clamp the discriminator weights. To do this, use `p.data.clamp_(-0.01, 0.01)`, where `p` is the discriminator weights.
- Hint 2: `n_critic` is the number of discriminator training iterations per generator training iteration.
- Hint 3: Use `X_tensor = torch.tensor(X_numpy, dtype=torch.float, device=DEVICE)` to convert numpy to tensor.

In [ ]:
class Fitter(object):
    def __init__(
        self,
        generator,
        discriminator,
        batch_size=32,
        n_epochs=10,
        latent_dim=1,
        lr=0.0001,
        n_critic=5,
    ):

        self.generator = generator
        self.discriminator = discriminator
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.latent_dim = latent_dim
        self.lr = lr
        self.n_critic = n_critic

        self.opt_gen = torch.optim.RMSprop(self.generator.parameters(), lr=self.lr)
        self.opt_disc = torch.optim.RMSprop(self.discriminator.parameters(), lr=self.lr)

        self.generator.to(DEVICE)
        self.discriminator.to(DEVICE)

    def fit(self, X, y):

        # numpy to tensor
        X_real = torch.tensor(X, dtype=torch.float, device=DEVICE)
        y_cond = torch.tensor(y, dtype=torch.float, device=DEVICE)

        # tensor to dataset
        dataset_real = TensorDataset(X_real, y_cond)

        # Turn on training
        self.generator.train(True)
        self.discriminator.train(True)

        self.loss_history = []

        # Fit GAN
        for epoch in range(self.n_epochs):
            for i, (real_batch, cond_batch) in enumerate(
                DataLoader(dataset_real, batch_size=self.batch_size, shuffle=True)
            ):

                ### YOUR CODE IS HERE ######


                ### THE END OF YOUR CODE ###

            # caiculate and store loss after an epoch
            Z_noise = torch.normal(0, 1, (len(X_real), self.latent_dim))
            X_fake = self.generator(Z_noise, y_cond)
            loss_epoch = torch.mean(self.discriminator(X_real, y_cond)) - torch.mean(
                self.discriminator(X_fake, y_cond)
            )
            self.loss_history.append(loss_epoch.detach().cpu())

        # Turn off training
        self.generator.train(False)
        self.discriminator.train(False)

## Training
We will train the model on the data.

In [ ]:
%%time
latent_dim = 10
generator = Generator(n_inputs=latent_dim + y.shape[1], n_outputs=X_train.shape[1])
discriminator = Discriminator(n_inputs=X_train.shape[1] + y.shape[1])

fitter = Fitter(
    generator,
    discriminator,
    batch_size=50,
    n_epochs=100,
    latent_dim=latent_dim,
    lr=0.0001,
    n_critic=5,
)
fitter.fit(X_train, y_train)

In [ ]:
# WGAN learning curve
plt.figure(figsize=(9, 5))
plt.plot(fitter.loss_history)
plt.xlabel("Epoch Number", size=14)
plt.ylabel("Loss Function", size=14)
plt.xticks(size=14)
plt.yticks(size=14)
plt.title("Conditional WGAN Learning Curve", size=14)
plt.grid(b=1, linestyle="--", linewidth=0.5, color="0.5")
plt.show()

## Task 5 (0.5 points)

Implement a function to generate new objects $X$ based on the vector of conditions $y$.

In [ ]:
def generate(generator, y, latent_dim):
    ### YOUR CODE IS HERE ######
    X_fake = ...
    ### THE END OF YOUR CODE ###
    return X_fake  # numpy

Now let's generate fake matrices `X_fake_train` and `X_fake_test`. Let's compare them with the matrices of real objects `X_train` and `X_test`.

In [ ]:
X_fake_train = generate(fitter.generator, y_train, latent_dim)

In [ ]:
plot_hists(X_train, X_fake_train, names, label1="Real", label2="Fake", bins=50)

In [ ]:
X_fake_test = generate(fitter.generator, y_test, latent_dim)

In [ ]:
plot_hists(X_test, X_fake_test, names, label1="Real", label2="Fake", bins=50)

# Measuring generation quality

<center><img src="img/clf.png" width="600"></center>

Let's measure the similarity of distributions using an external classifier (think about why).

In [ ]:
# combining real and fake matrices into one
XX_train = np.concatenate((X_fake_train, X_train), axis=0)
XX_test = np.concatenate((X_fake_test, X_test), axis=0)

yy_train = np.array([0] * len(X_fake_train) + [1] * len(X_train))
yy_test = np.array([0] * len(X_fake_test) + [1] * len(X_test))

In [ ]:
# training the classifier
clf = GradientBoostingClassifier()
clf.fit(XX_train, yy_train)

# get forecasts
yy_test_proba = clf.predict_proba(XX_test)[:, 1]

In [ ]:
auc = roc_auc_score(yy_test, yy_test_proba)
print("ROC AUC = ", auc)

# Conditional Variational Autoencoders

<center><img src="img/cvae.svg" width="600"></center>

Now, let's solve the same problem using a conditional autoencoder (CVAE).

## Task 6 (0.5 points)

Implement a neural network for an encoder with the following layers:
- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Output layer for mu; Output layer for log_sigma;

Hint: use the `nn.Sequential()` function.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_inputs, lat_size):
        super().__init__()

        ### YOUR CODE IS HERE ######
        self.enc_net = ...

        self.mu = ...
        self.log_sigma = ...
        ### THE END OF YOUR CODE ###

    def forward(self, x, y):
        z = torch.cat((x, y), dim=1)
        z = self.enc_net(z)
        mu = self.mu(z)
        log_sigma = self.log_sigma(z)
        return mu, log_sigma

## Task 7 (0.5 points)

Implement a neural network for a decoder with the following layers:
- Fully connected layer with 100 neurons;
- ReLU activation function;
- Fully connected layer with 100 neurons;
- ReLU activation function;
- Output layer.

Hint: use the `nn.Sequential()` function.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()

        ### YOUR CODE IS HERE ######
        self.dec_net = ...
        ### THE END OF YOUR CODE ###

    def forward(self, z, y):
        z_cond = torch.cat((z, y), dim=1)
        x_rec = self.dec_net(z_cond)
        return x_rec

## Task 8 (0.5 points)

Implement a class for training a variational autoencoder.

In [ ]:
class VAEFitter:
    def __init__(
        self,
        encoder,
        decoder,
        batch_size=32,
        n_epochs=10,
        latent_dim=1,
        lr=0.0001,
        KL_weight=0.001,
    ):

        self.encoder = encoder
        self.decoder = decoder
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.latent_dim = latent_dim
        self.lr = lr
        self.KL_weight = KL_weight

        self.criterion = nn.MSELoss()
        self.opt = torch.optim.RMSprop(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=self.lr,
        )

        self.encoder.to(DEVICE)
        self.decoder.to(DEVICE)

    def sample_z(self, mu, log_sigma):
        eps = torch.randn(mu.shape).to(DEVICE)
        return mu + torch.exp(log_sigma / 2) * eps

    def custom_loss(self, x, rec_x, mu, log_sigma):
        KL = torch.mean(
            -0.5 * torch.sum(1 + log_sigma - mu**2 - log_sigma.exp(), dim=1), dim=0
        )
        recon_loss = self.criterion(x, rec_x)
        return KL * self.KL_weight + recon_loss

    def compute_loss(self, x_batch, cond_batch):

        ### YOUR CODE IS HERE ######
        loss = ...
        ### THE END OF YOUR CODE ###

        return loss

    def fit(self, X, y):

        # numpy to tensor
        X_real = torch.tensor(X, dtype=torch.float, device=DEVICE)
        y_cond = torch.tensor(y, dtype=torch.float, device=DEVICE)

        # tensor to dataset
        dataset_real = TensorDataset(X_real, y_cond)

        # Turn on training
        self.encoder.train(True)
        self.decoder.train(True)

        self.loss_history = []

        # Fit GAN
        for epoch in range(self.n_epochs):
            for i, (x_batch, cond_batch) in enumerate(
                DataLoader(dataset_real, batch_size=self.batch_size, shuffle=True)
            ):

                # caiculate loss
                loss = self.compute_loss(x_batch, cond_batch)

                # optimization step
                self.opt.zero_grad()
                loss.backward()
                self.opt.step()

            # caiculate and store loss after an epoch
            loss_epoch = self.compute_loss(X_real, y_cond)
            self.loss_history.append(loss_epoch.detach().cpu())

        # Turn off training
        self.encoder.train(False)
        self.decoder.train(False)

## Training
We will train the model on the data.

In [ ]:
%%time

latent_dim = 10

encoder = Encoder(n_inputs=X_train.shape[1] + y.shape[1], lat_size=latent_dim)
decoder = Decoder(n_inputs=latent_dim + y.shape[1], n_outputs=X_train.shape[1])

vae_fitter = VAEFitter(
    encoder,
    decoder,
    batch_size=50,
    n_epochs=100,
    latent_dim=latent_dim,
    lr=0.001,
    KL_weight=0.001,
)
vae_fitter.fit(X_train, y_train)

In [ ]:
# WGAN learning curve
plt.figure(figsize=(9, 5))
plt.plot(vae_fitter.loss_history)
plt.xlabel("Epoch Number", size=14)
plt.ylabel("Loss Function", size=14)
plt.xticks(size=14)
plt.yticks(size=14)
plt.title("Conditional VAE Learning Curve", size=14)
plt.grid(b=1, linestyle="--", linewidth=0.5, color="0.5")
plt.show()

## Task 9 (0.5 points)

Implement a function to generate new objects $X$ based on the vector of conditions $y$.

In [ ]:
def generate(decoder, y, latent_dim):
    ### YOUR CODE IS HERE ######
    X_fake = ...
    ### THE END OF YOUR CODE ###
    return X_fake  # numpy

Now let's generate fake matrices `X_fake_train` and `X_fake_test`. Let's compare them with the matrices of real objects `X_train` and `X_test`.

In [ ]:
X_fake_train = generate(vae_fitter.decoder, y_train, latent_dim)

In [ ]:
plot_hists(X_train, X_fake_train, names, label1="Real", label2="Fake", bins=50)

In [ ]:
X_fake_test = generate(vae_fitter.decoder, y_test, latent_dim)

In [ ]:
plot_hists(X_test, X_fake_test, names, label1="Real", label2="Fake", bins=50)

# Measuring generation quality

Let's measure the similarity of distributions using a classifier.

In [ ]:
# combining real and fake matrices into one
XX_train = np.concatenate((X_fake_train, X_train), axis=0)
XX_test = np.concatenate((X_fake_test, X_test), axis=0)

yy_train = np.array([0] * len(X_fake_train) + [1] * len(X_train))
yy_test = np.array([0] * len(X_fake_test) + [1] * len(X_test))

In [ ]:
# training the classifier
clf = GradientBoostingClassifier()
clf.fit(XX_train, yy_train)

# get forecasts
yy_test_proba = clf.predict_proba(XX_test)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(yy_test, yy_test_proba)
print("ROC AUC = ", auc)

# Diffusion models

Same as above, but now diffusion models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 10 (0.5 points)

Implement a function for adding noise to data, adapting it to our data type.

In [ ]:
def corrupt(x, amount):
    ### YOUR CODE IS HERE ######
    x = ...
    ### THE END OF YOUR CODE ###
    return x

Tip: read the documentation on the scheduler, as it may not be optimal for our data as described below (because we don't have images!) :)

In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
plt.plot(
    noise_scheduler.alphas_cumprod.cpu() ** 0.5, label=r"${\sqrt{\bar{\alpha}_t}}$"
)
plt.plot(
    (1 - noise_scheduler.alphas_cumprod.cpu()) ** 0.5,
    label=r"$\sqrt{(1 - \bar{\alpha}_t)}$",
)
plt.legend(fontsize="x-large")

## Task 11 (0.5 points)

Implement a neural network. You can use the generator model as the architecture. During the experiments, try changing the model architecture to improve the quality of the generated objects.

- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Fully connected layer with 100 neurons;
- Batch normalization layer;
- ReLU activation function;
- Output layer.

In [ ]:
class DiffusionGenerator(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()

        ### YOUR CODE IS HERE ######

        ### THE END OF YOUR CODE ###

    def forward(self, z, y):
        zy = torch.cat((z, y), dim=1)
        return ...

## Task 12 (0.5 points)

Write a function to generate a new object using the trained model.

In [ ]:
def generate_with_diffusion(model, y, latent_dim, sheduler):
    ### YOUR CODE IS HERE ######
    X_fake = ...
    ### THE END OF YOUR CODE ###
    return X_fake  # numpy

## Task 13 (1 point)

Write a training class for the diffusion model and train the model, then describe the results obtained. You can change some parts of the code for your convenience, but please leave comments in such cases.

In [ ]:
class DiffusionFitter:
    def __init__(
        self,
        model,
        batch_size=32,
        n_epochs=10,
        latent_dim=1,
        lr=0.0001,
        n_critic=5,
    ):

        self.model = model
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.latent_dim = latent_dim
        self.lr = lr
        self.n_critic = n_critic

        self.opt_gen = torch.optim.RMSprop(self.model.parameters(), lr=self.lr)

        self.model.to(DEVICE)

    def fit(self, X, y):

        # numpy to tensor
        X_real = torch.tensor(X, dtype=torch.float, device=DEVICE)
        y_cond = torch.tensor(y, dtype=torch.float, device=DEVICE)

        # tensor to dataset
        dataset_real = TensorDataset(X_real, y_cond)

        # Turn on training
        self.model.train(True)

        self.loss_history = []

        # Fit
        for epoch in range(self.n_epochs):
            loss_epoch = 0
            for i, (real_batch, cond_batch) in enumerate(
                DataLoader(dataset_real, batch_size=self.batch_size, shuffle=True)
            ):

                ### YOUR CODE IS HERE ######

                ...

                loss_epoch += ...

                ### THE END OF YOUR CODE ###

            # caiculate and store loss after an epoch

            self.loss_history.append(loss_epoch)

        # Turn off training
        self.model.train(False)

In [ ]:
%%time
latent_dim = 10
model = DiffusionGenerator(n_inputs=latent_dim + y.shape[1], n_outputs=X_train.shape[1])

diffusionFitter = DiffusionFitter(
    model,
    batch_size=50,
    n_epochs=100,
    latent_dim=latent_dim,
    lr=0.0001,
    n_critic=5,
)
diffusionFitter.fit(X_train, y_train)

In [ ]:
# diffusion learning curve
plt.figure(figsize=(9, 5))
plt.plot(diffusionFitter.loss_history)
plt.xlabel("Epoch Number", size=14)
plt.ylabel("Loss Function", size=14)
plt.xticks(size=14)
plt.yticks(size=14)
plt.title("Conditional diffusing model Learning Curve", size=14)
plt.grid(b=1, linestyle="--", linewidth=0.5, color="0.5")
plt.show()

## Task 14 (0.5 points)
Similar to previous experiments with the GAN model, generate a sample of fake objects equal to the size of the test sample and train gradient boosting. Train the model to distinguish real objects from fake ones, then calculate the ROC-AUC metric. What were the results? How do you evaluate them? How do they compare to the sWGAN model?

# Normalization flows

## Task 15 (1 point)

Diffusion has proven itself to be a worthy competitor to the GAN model. Since there is not much data, training did not take long, and the task is not complicated, the differences from GAN are not so noticeable, but still noteworthy.

Let's try to train RealNVP to solve this task.

**Add the necessary to the base class.**

In [ ]:
trainloader = torch.utils.data.DataLoader(X_train, batch_size=64, shuffle=True)

In [ ]:
# Main class for NormFlow
class NormalizingFlow(nn.Module):

    def __init__(self, layers, prior):
        super(NormalizingFlow, self).__init__()

        # your code below

    def log_prob(self, x):
        log_likelihood = None

        for layer in self.layers:
            x, change = layer.f(x)
            if log_likelihood is not None:
                log_likelihood = # your code here
            else:
                log_likelihood = # your code here

        log_likelihood = # your code here

        return log_likelihood.mean()

    def sample(self, num_samples):
        x = self.prior.sample((num_samples, ))

        for layer in self.layers[::-1]:
            x = layer.g(x)

        return x

## Task 16 (1 point)

Implement the RealNVP neural network. Use a neural network (function) with the following parameters for forward and backward transformation:

- Fully connected layer with 100 neurons;
- ReLU activation function;
- Output layer.

In [ ]:
import torch.nn as nn

class RealNVP(nn.Module):
    def __init__(self, var_size, mask, hidden=100):
        super(RealNVP, self).__init__()
        self.mask = mask  # You may not need this. Take it as a hint.
        self.var_size = var_size

        self.nn_t = # your code here
        self.nn_s = # your code here

    def f(self, x):
        t = # your code here
        s = # your code here

        new_x = # your code here

        log_det = # your code here
        return new_x, log_det

    def g(self, x):
        t = # your code here
        s = # your code here

        new_x = # your code here
        return new_x

In [ ]:
def train_nf(tr_dataloader, nf, opt, num_epochs):
    nf.train()
    loss_trace = []

    iter_i = 0

    for epoch_i in range(num_epochs):
        print(f'Epoch {epoch_i + 1}')
        for batch in tr_dataloader:

            x = batch.float()

            opt.zero_grad()

            loss = # your code here
            loss.backward()

            opt.step()

            loss_trace.append((iter_i, loss.item()))

            iter_i += 1

In [ ]:
prior = torch.distributions.MultivariateNormal(torch.zeros(10), torch.eye(10))

layers = []
for i in range(4):
    layers.append(RealNVP(var_size=10, mask=((torch.arange(10) + i) % 2)))

nf = NormalizingFlow(layers=layers, prior=prior)

opt = torch.optim.Adam(nf.parameters(), lr=1e-3)

In [ ]:
train_nf(trainloader, nf, opt, num_epochs=10)

## Task 17 (0.5 points)

Similar to the previous diffusion experiment, generate a sample of fake objects equal to the size of the test sample and train gradient boosting. Train the model to distinguish real objects from fake ones, then calculate the ROC-AUC. What were the results? How do you evaluate them? How do they compare to other models?

# Improvements (1+ points)

Try adjusting the training parameters of a model or improving them in some other way to get the lowest possible ROC AUC. What did you get? Which model is better?

We award 0.1 points for every hundredth of a point above (below) ROC-AUC=0.65, not inclusive. That is, for 0.65 you get 0, for 0.649 -- 0.1, 0.639 -- 0.2, 0.609 -- 0.5, 0.559 -- 1.

As is well known, in binary classification, it is impossible to achieve a value higher than 0.5.

In [ ]:
# your code here